# Q3: Feature Engineering and Regression Pipeline

This notebook builds a reproducible scikit-learn regression pipeline to predict `items_sold` at a retail store using the `q3_retail_promotions.csv` dataset.

## 1. Date Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/q3_retail_promotions.csv')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Date range:', df['transaction_date'].min(), 'to', df['transaction_date'].max())
print('\nPromotion types:', df['promotion_type'].unique().tolist())
print('Location types:', df['location_type'].unique().tolist())
print('Store sizes:', df['store_size'].unique().tolist())

Shape: (1200, 9)
Columns: ['transaction_date', 'store_id', 'store_size', 'location_type', 'promotion_type', 'is_weekend', 'is_festival', 'competition_density', 'items_sold']
Date range: 2022-01-01 to 2024-12-31

Promotion types: ['free_gift', 'loyalty_points', 'bogo', 'category_offer', 'flat_discount']
Location types: ['semi-urban', 'urban', 'rural']
Store sizes: ['small', 'medium', 'large']


In [2]:
# Extract date features from transaction_date
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)  # 1 if day >= 25

print('New columns added: year, month, day_of_week, is_month_end')
print('\nSample of engineered features:')
print(df[['transaction_date', 'year', 'month', 'day_of_week', 'is_month_end']].head(10))
print(f'\nRows with is_month_end=1: {df["is_month_end"].sum()} ({df["is_month_end"].mean()*100:.1f}%)')

New columns added: year, month, day_of_week, is_month_end

Sample of engineered features:
  transaction_date  year  month  day_of_week  is_month_end
0       2022-01-01  2022      1            5             0
1       2022-01-01  2022      1            5             0
2       2022-01-02  2022      1            6             0
3       2022-01-02  2022      1            6             0
4       2022-01-03  2022      1            0             0
5       2022-01-03  2022      1            0             0
6       2022-01-04  2022      1            1             0
7       2022-01-05  2022      1            2             0
8       2022-01-06  2022      1            3             0
9       2022-01-07  2022      1            4             0

Rows with is_month_end=1: 246 (20.5%)


## 2. Temporal Train-Test Split

In [3]:
# Sort by date and use most recent 20% as test
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.8)

train = df_sorted.iloc[:split_idx].copy()
test  = df_sorted.iloc[split_idx:].copy()

print(f'Train set: {train.shape[0]} rows')
print(f'  Date range: {train["transaction_date"].min().date()} → {train["transaction_date"].max().date()}')
print(f'Test set:  {test.shape[0]} rows')
print(f'  Date range: {test["transaction_date"].min().date()} → {test["transaction_date"].max().date()}')

Train set: 960 rows
  Date range: 2022-01-01 → 2024-06-11
Test set:  240 rows
  Date range: 2024-06-12 → 2024-12-31


**Why a random split is inappropriate for time-ordered data:**

A random 80/20 split would scatter test samples throughout the time series — the model would be trained on future data and tested on past data, creating **data leakage**. In the real world, a deployed model only has access to historical data at the time of prediction. A temporal split — where the test set is strictly the most recent observations — correctly simulates this: the model learns from the past and is evaluated on genuinely unseen future records. Violating this leads to overly optimistic evaluation metrics that do not reflect real deployment performance.

## 3. Preprocessing Pipeline

In [4]:
# Define feature groups
cat_feats = ['promotion_type', 'location_type', 'store_size']
num_feats = ['competition_density', 'is_weekend', 'is_festival',
             'year', 'month', 'day_of_week', 'is_month_end']

drop_cols = ['transaction_date', 'store_id']
X_train = train.drop(columns=drop_cols + ['items_sold'])
y_train = train['items_sold']
X_test  = test.drop(columns=drop_cols + ['items_sold'])
y_test  = test['items_sold']

# Build ColumnTransformer preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats),
    ('num', StandardScaler(), num_feats)
], remainder='drop')

print('Pipeline preprocessor built.')
print(f'  Categorical features: {cat_feats}')
print(f'  Numerical features:   {num_feats}')
print('  The pipeline will be fit ONLY on train set and applied to both train and test.')

Pipeline preprocessor built.
  Categorical features: ['promotion_type', 'location_type', 'store_size']
  Numerical features:   ['competition_density', 'is_weekend', 'is_festival', 'year', 'month', 'day_of_week', 'is_month_end']
  The pipeline will be fit ONLY on train set and applied to both train and test.


## 4. Model Training and Evaluation

In [5]:
# Linear Regression Pipeline
lr_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)

# Random Forest Regressor Pipeline
rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])
rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)

print(f'{'Model':<22} {'RMSE':>8} {'MAE':>8}')
print('-' * 40)
print(f'{'Linear Regression':<22} {rmse_lr:>8.2f} {mae_lr:>8.2f}')
print(f'{'Random Forest':<22} {rmse_rf:>8.2f} {mae_rf:>8.2f}')

Model                       RMSE      MAE
----------------------------------------
Linear Regression          27.13    21.07
Random Forest              31.22    24.96


In [6]:
# Parity plots: predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, title in zip(axes, [y_pred_lr, y_pred_rf],
                              ['Linear Regression', 'Random Forest Regressor']):
    ax.scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=30)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Perfect Prediction')
    ax.set_title(f'{title} — Parity Plot', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual items_sold')
    ax.set_ylabel('Predicted items_sold')
    ax.legend()

plt.tight_layout()
plt.savefig('parity_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print('Parity plots saved.')

Parity plots saved.


In [7]:
# Feature importances from Random Forest
ohe = rf_pipe.named_steps['prep'].named_transformers_['cat']
ohe_feature_names = ohe.get_feature_names_out(cat_feats)
all_feature_names = list(ohe_feature_names) + num_feats
importances = rf_pipe.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)
print('Top 5 Most Influential Features:')
print(feat_imp.head(5).to_string())

# Plot top 10
plt.figure(figsize=(10, 5))
feat_imp.head(10).sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Random Forest — Top 10 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Feature importance plot saved.')

Top 5 Most Influential Features:
is_festival            0.173540
store_size_small       0.168274
location_type_urban    0.109020
day_of_week            0.084995
is_weekend             0.069907

Feature importance plot saved.


**Model Performance Summary:**

- **Linear Regression** achieved RMSE=27.13 and MAE=21.07 — outperforming Random Forest (RMSE=31.22, MAE=24.96) on this test set.
- The better Linear Regression performance suggests the relationship between features and `items_sold` is largely linear in this dataset, and the Random Forest may be slightly overfitting to training patterns that don't generalise to the future test period.

**Top 5 Most Influential Features (from Random Forest):**
1. **`is_festival` (0.17):** Festival periods drive the largest spikes in items sold — the single most powerful predictor.
2. **`store_size_small` (0.17):** Store size has a major impact on sales volume — small stores have distinctly different sales dynamics.
3. **`location_type_urban` (0.11):** Urban stores have higher footfall and different purchasing patterns.
4. **`day_of_week` (0.08):** Sales vary significantly across weekdays vs. weekends.
5. **`is_weekend` (0.07):** Weekend shopping shows a distinct uplift in items sold.

Notably, `promotion_type` features appear further down the importance list, suggesting that store characteristics and calendar effects have more influence than the specific promotion run.